Notebook ran in local setup

In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install transformers==4.40.2
!pip install peft==0.10.0
!pip install accelerate==0.30.1
!pip install bitsandbytes==0.43.1
!pip install datasets huggingface_hub

Looking in indexes: https://download.pytorch.org/whl/cu121



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


start here

In [3]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from huggingface_hub import login
login()

In [5]:
!nvidia-smi -L

GPU 0: NVIDIA GeForce RTX 4060 Laptop GPU (UUID: GPU-b88f4c44-dca5-1f26-1b00-a9f7bccae5ee)


In [6]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1️⃣ Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-1.3b")
tokenizer.pad_token = tokenizer.eos_token

# 2️⃣ Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# 3️⃣ Load model
model = AutoModelForCausalLM.from_pretrained(
    "facebook/opt-1.3b",
    quantization_config=bnb_config,
    device_map="auto"
)

d:\python\lora, qlora\peft\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
d:\python\lora, qlora\peft\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
import torch.nn as nn

In [8]:
for param in model.parameters():
  param.requires_grad = False  # freeze the model - train adapters later
  if param.ndim == 1:
    # cast the small parameters (e.g. layernorm) to fp32 for stability
    param.data = param.data.to(torch.float32)

model.gradient_checkpointing_enable()  # reduce number of stored activations
model.enable_input_require_grads()

class CastOutputToFloat(nn.Sequential):
  def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

In [9]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

![alt text](image.png)

In [10]:
from peft import LoraConfig, get_peft_model 

config = LoraConfig(
    r=16, #attention heads
    lora_alpha=32, #alpha scaling
    # target_modules=["q_proj", "v_proj"], #if you know the 
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # set this for CLM or Seq2Seq
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 3145728 || all params: 714924032 || trainable%: 0.4400087085056892


data

In [11]:
import transformers
from datasets import load_dataset
data = load_dataset("Abirate/english_quotes")


In [12]:
def merge_columns(example):
    example["prediction"] = example["quote"] + " ->: " + str(example["tags"])
    return example

data['train'] = data['train'].map(merge_columns)
data['train']["prediction"][:5]

["“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']",
 "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.” ->: ['best', 'life', 'love', 'mistakes', 'out-of-control', 'truth', 'worst']",
 "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.” ->: ['human-nature', 'humor', 'infinity', 'philosophy', 'science', 'stupidity', 'universe']",
 "“So many books, so little time.” ->: ['books', 'humor']",
 "“A room without books is like a body without a soul.” ->: ['books', 'simile', 'soul']"]

In [13]:
data['train'][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'prediction': "“Be yourself; everyone else is already taken.” ->: ['be-yourself', 'gilbert-perreira', 'honesty', 'inspirational', 'misattributed-oscar-wilde', 'quote-investigator']"}

In [14]:
data = data.map(lambda samples: tokenizer(samples['prediction']), batched=True)

training

In [15]:

trainer = transformers.Trainer(
    model=model, 
    train_dataset=data['train'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1, 
        gradient_accumulation_steps=8,
        warmup_steps=100, 
        max_steps=200, 
        learning_rate=2e-4, 
        fp16=True,
        logging_steps=1, 
        output_dir='outputs'
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
trainer.train()

d:\python\lora, qlora\peft\lib\site-packages\accelerate\accelerator.py:479: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


  0%|          | 0/200 [00:00<?, ?it/s]

d:\python\lora, qlora\peft\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]


{'loss': 3.4801, 'grad_norm': 2.803436279296875, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.0}
{'loss': 3.442, 'grad_norm': 3.052739143371582, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.01}
{'loss': 3.7876, 'grad_norm': 3.2427775859832764, 'learning_rate': 6e-06, 'epoch': 0.01}
{'loss': 3.4554, 'grad_norm': 2.2913646697998047, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.01}
{'loss': 3.6159, 'grad_norm': 2.4836721420288086, 'learning_rate': 1e-05, 'epoch': 0.02}
{'loss': 2.9386, 'grad_norm': 2.2348504066467285, 'learning_rate': 1.2e-05, 'epoch': 0.02}
{'loss': 3.5443, 'grad_norm': 2.3041791915893555, 'learning_rate': 1.4000000000000001e-05, 'epoch': 0.02}
{'loss': 3.4804, 'grad_norm': 2.6674585342407227, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.03}
{'loss': 2.951, 'grad_norm': 1.9879926443099976, 'learning_rate': 1.8e-05, 'epoch': 0.03}
{'loss': 3.1568, 'grad_norm': 2.412905693054199, 'learning_rate': 2e-05, 'epoch': 0.03}
{'loss': 3.5562, 'grad_norm':

TrainOutput(global_step=200, training_loss=2.318014245033264, metrics={'train_runtime': 397.4674, 'train_samples_per_second': 4.025, 'train_steps_per_second': 0.503, 'total_flos': 692986765025280.0, 'train_loss': 2.318014245033264, 'epoch': 0.6379585326953748})

push to HF

In [16]:
model.push_to_hub("saiproj592/facebook-opt-1.3b-lora-tagger",
                  use_auth_token=True,
                  commit_message="basic training",
                  private=True)

d:\python\lora, qlora\peft\lib\site-packages\transformers\utils\hub.py:834: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
d:\python\lora, qlora\peft\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/saiproj592/facebook-opt-1.3b-lora-tagger/commit/7c3094556326cc179dee6833114e3da41e5221e8', commit_message='basic training', commit_description='', oid='7c3094556326cc179dee6833114e3da41e5221e8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/saiproj592/facebook-opt-1.3b-lora-tagger', endpoint='https://huggingface.co', repo_type='model', repo_id='saiproj592/facebook-opt-1.3b-lora-tagger'), pr_revision=None, pr_num=None)

load from HF

In [18]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

peft_model_id = "saiproj592/facebook-opt-1.3b-lora-tagger"
config = PeftConfig.from_pretrained(peft_model_id)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, return_dict=True, load_in_8bit=True, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# Load the Lora model
model = PeftModel.from_pretrained(model, peft_model_id)

adapter_config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

d:\python\lora, qlora\peft\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\prasanth\.cache\huggingface\hub\models--saiproj592--facebook-opt-1.3b-lora-tagger. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future v

adapter_model.safetensors:   0%|          | 0.00/12.6M [00:00<?, ?B/s]

inference

In [22]:
batch = tokenizer("“Training models with PEFT and LoRa is cool” ->: ", return_tensors='pt')

with torch.cuda.amp.autocast():
  output_tokens = model.generate(**batch, max_new_tokens=50)

print('\n\n', tokenizer.decode(output_tokens[0], skip_special_tokens=True))

C:\Users\prasanth\AppData\Local\Temp\ipykernel_37764\659789896.py:3: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():




 “Training models with PEFT and LoRa is cool” ->: ˈtrɒnəm/ [trɒnəm] [trɒnəm] ->: ['training','models', 'PEFT', 'LoRa'] ->: ['training','models
